# Word Embeddings

Bag-of-words and TF-IDF represent words as sparse, high-dimensional vectors with no notion of similarity — *"cat"* and *"dog"* are as unrelated as *"cat"* and *"economics"*. **Word embeddings** map each word to a dense, low-dimensional vector such that **semantically similar words are close together** in vector space.

This was a breakthrough in NLP. Embeddings capture meaning from the **distributional hypothesis** — words appearing in similar contexts tend to have similar meanings. *"The cat sat on the mat"* and *"The dog sat on the rug"* teach the model that *"cat"* and *"dog"* play similar roles, and *"mat"* and *"rug"* are related.

This notebook covers the motivation for embeddings, how they are learned (Word2Vec, GloVe), vector arithmetic, similarity search, using embeddings for downstream tasks, and the shift to contextual embeddings in modern language models.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. The Problem with Sparse Representations

One-hot and bag-of-words vectors have dimension equal to vocabulary size ($V$ can be 100,000+). Each word occupies one axis — orthogonal to every other word.

| Issue | Consequence |
|-------|-------------|
| **High dimensionality** | Memory and compute scale with $V$ |
| **Sparsity** | Most entries are zero; little information per dimension |
| **No similarity** | $\text{sim}(\text{cat}, \text{dog}) = 0$ in one-hot space |
| **No generalization** | *"puppy"* and *"dog"* share no features; OOV words are invisible |

Embeddings solve this by learning compact vectors (typically 50–300 dimensions) where geometric relationships encode semantic relationships.

In [ ]:
vocab = ["cat", "dog", "car", "economics"]
word2idx = {w: i for i, w in enumerate(vocab)}

def one_hot(word, size=len(vocab)):
    vec = np.zeros(size)
    vec[word2idx[word]] = 1
    return vec

print("One-hot vectors (each word is orthogonal):")
for w in vocab:
    print(f"  {w:12s} {one_hot(w).astype(int)}")

sim = cosine_similarity([one_hot("cat")], [one_hot("dog")])[0, 0]
print(f"\nCosine similarity(cat, dog) with one-hot: {sim:.1f}")

---

## 2. What Are Word Embeddings?

A **word embedding** is a learned mapping from words to dense vectors $\mathbf{e}_w \in \mathbb{R}^d$ where $d \ll V$ (typically $d = 100$ to $300$).

Properties of good embeddings:
- **Similar words are nearby** — *"happy"* and *"joyful"* have high cosine similarity.
- **Linear structure captures relationships** — $\mathbf{e}_{\text{king}} - \mathbf{e}_{\text{man}} + \mathbf{e}_{\text{woman}} \approx \mathbf{e}_{\text{queen}}$.
- **Clustering by topic** — animal names group together; country names group together.
- **Transferable** — pre-trained embeddings improve downstream tasks with limited data.

Embeddings are stored in an **embedding matrix** $\mathbf{E} \in \mathbb{R}^{V \times d}$ where row $i$ is the vector for word $i$. Neural networks look up embeddings by index — an operation called an **embedding layer**.

---

## 3. The Distributional Hypothesis

> *"You shall know a word by the company it keeps."* — J.R. Firth, 1957

The **distributional hypothesis** states that words appearing in similar contexts have similar meanings. Consider:

- *"I drank ___ in the morning."* → coffee, tea, water (beverages)
- *"The ___ barked loudly."* → dog, puppy, hound (canines)

By observing millions of such contexts, embedding algorithms learn that *"coffee"* and *"tea"* should be close (both fill beverage slots) while *"coffee"* and *"barked"* should be far apart.

This is **self-supervised learning** — no manual labels needed. The context words themselves serve as training signal.

---

## 4. Co-occurrence Matrices

A foundational approach: build a **word-context co-occurrence matrix** $\mathbf{M}$ where $M_{ij}$ counts how often word $i$ appears near word $j$ (within a window of $k$ words).

The matrix is dense with semantic information but still high-dimensional ($V \times V$). **Dimensionality reduction** (SVD) compresses it into low-dimensional embeddings — this is the basis of **Latent Semantic Analysis (LSA)**.

In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the mouse",
    "the dog chased the cat",
    "a puppy played with a kitten",
    "the kitten played on the mat",
    "a car drove down the road",
    "the truck drove on the road",
]

def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

def build_cooccurrence(corpus, window_size=2):
    vocab = sorted(set(w for doc in corpus for w in tokenize(doc)))
    word2idx = {w: i for i, w in enumerate(vocab)}
    matrix = np.zeros((len(vocab), len(vocab)))

    for doc in corpus:
        tokens = tokenize(doc)
        for i, target in enumerate(tokens):
            ti = word2idx[target]
            start = max(0, i - window_size)
            end = min(len(tokens), i + window_size + 1)
            for j in range(start, end):
                if j != i:
                    matrix[ti, word2idx[tokens[j]]] += 1
    return matrix, vocab

cooc, vocab = build_cooccurrence(corpus, window_size=2)
df_cooc = pd.DataFrame(cooc, index=vocab, columns=vocab)
print("Co-occurrence matrix (window=2):")
print(df_cooc.astype(int).to_string())

---

## 5. Learning Embeddings via SVD

Apply **Singular Value Decomposition** to the co-occurrence matrix and keep the top $d$ singular vectors:

$$\mathbf{M} \approx \mathbf{U}_d \mathbf{\Sigma}_d \mathbf{V}_d^\top$$

The rows of $\mathbf{U}_d \mathbf{\Sigma}_d^{1/2}$ (or $\mathbf{M} \mathbf{V}_d$) serve as word embeddings. Words that co-occur with similar neighbors end up with similar reduced vectors.

In [ ]:
def svd_embeddings(cooc_matrix, dim=2):
    svd = TruncatedSVD(n_components=dim, random_state=42)
    embeddings = svd.fit_transform(cooc_matrix)
    return embeddings, svd.explained_variance_ratio_.sum()

embeddings, variance = svd_embeddings(cooc, dim=2)
print(f"Embedding shape: {embeddings.shape}")
print(f"Variance explained (2D): {variance:.2%}\n")

for word, vec in zip(vocab, embeddings):
    print(f"  {word:10s} [{vec[0]:+.3f}, {vec[1]:+.3f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for word, vec in zip(vocab, embeddings):
    ax.scatter(vec[0], vec[1], s=80, color="steelblue")
    ax.annotate(word, (vec[0], vec[1]), textcoords="offset points",
                xytext=(5, 5), fontsize=10)
ax.set_xlabel("Dimension 1")
ax.set_ylabel("Dimension 2")
ax.set_title("Word embeddings from co-occurrence + SVD")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 6. Word2Vec

**Word2Vec** (Mikolov et al., 2013) learns embeddings directly with a shallow neural network, avoiding the explicit $V \times V$ co-occurrence matrix. Two architectures:

**Skip-gram:** Given a center word, predict surrounding context words.

```
Input: "sat"  →  Predict: "the", "on", "the", "mat"
```

**CBOW (Continuous Bag of Words):** Given surrounding context, predict the center word.

```
Input: "the", "on", "the", "mat"  →  Predict: "sat"
```

Skip-gram works better for rare words; CBOW trains faster. Both learn an embedding matrix as the first layer of the network.

**Negative sampling** makes training tractable. Instead of computing softmax over the entire vocabulary (expensive), the model updates the center word embedding to be close to a true context word and far from a few randomly sampled "negative" words.

---

## 7. GloVe

**GloVe** (Global Vectors, Pennington et al., 2014) combines global co-occurrence statistics with local context prediction. It factorizes a **log co-occurrence matrix** where rare co-occurrences are downweighted:

$$\mathbf{w}_i^\top \mathbf{w}_j + b_i + b_j = \log X_{ij}$$

GloVe and Word2Vec produce embeddings of similar quality. GloVe leverages global corpus statistics; Word2Vec learns from local context windows. Both capture semantic and syntactic relationships.

---

## 8. Cosine Similarity

Embeddings are compared using **cosine similarity** — the cosine of the angle between two vectors:

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

Values range from $-1$ (opposite) to $+1$ (identical direction). For normalized embeddings, cosine similarity equals the dot product. It measures semantic closeness regardless of vector magnitude.

In [ ]:
def most_similar(word, embeddings, vocab, top_n=5):
    idx = vocab.index(word)
    sims = cosine_similarity([embeddings[idx]], embeddings)[0]
    ranked = np.argsort(sims)[::-1]
    results = []
    for i in ranked:
        if vocab[i] != word:
            results.append((vocab[i], sims[i]))
        if len(results) >= top_n:
            break
    return results

for query in ["cat", "dog", "car"]:
    print(f"Words most similar to '{query}':")
    for w, sim in most_similar(query, embeddings, vocab, top_n=4):
        print(f"  {w:10s} {sim:.3f}")
    print()

---

## 9. Embedding Arithmetic

Word2Vec embeddings famously support **vector arithmetic**:

$$\mathbf{e}_{\text{king}} - \mathbf{e}_{\text{man}} + \mathbf{e}_{\text{woman}} \approx \mathbf{e}_{\text{queen}}$$

The difference $\mathbf{e}_{\text{king}} - \mathbf{e}_{\text{man}}$ captures the concept of "royalty applied to male → female." Adding it to $\mathbf{e}_{\text{woman}}$ lands near $\mathbf{e}_{\text{queen}}$.

Similar patterns exist for capitals, verb tenses, and comparatives. The arithmetic works because embeddings encode relational structure in linear subspaces.

In [ ]:
# Illustrative 50-dimensional embeddings (structured for analogy demo)
analogy_vocab = ["king", "queen", "man", "woman", "prince", "princess",
                 "boy", "girl", "paris", "france", "london", "england"]
dim = 50
rng = np.random.RandomState(42)

gender = rng.randn(dim) * 0.3
royalty = rng.randn(dim) * 0.3
country = rng.randn(dim) * 0.3
city = rng.randn(dim) * 0.3
base = rng.randn(dim) * 0.1

analogy_embeddings = {}
analogy_embeddings["king"] = base + royalty + gender
analogy_embeddings["queen"] = base + royalty - gender
analogy_embeddings["man"] = base + gender
analogy_embeddings["woman"] = base - gender
analogy_embeddings["prince"] = base + royalty + gender + rng.randn(dim) * 0.05
analogy_embeddings["princess"] = base + royalty - gender + rng.randn(dim) * 0.05
analogy_embeddings["boy"] = base + gender + rng.randn(dim) * 0.05
analogy_embeddings["girl"] = base - gender + rng.randn(dim) * 0.05
analogy_embeddings["france"] = base + country
analogy_embeddings["paris"] = base + country + city
analogy_embeddings["england"] = base + country + rng.randn(dim) * 0.1
analogy_embeddings["london"] = base + country + city + rng.randn(dim) * 0.1

def get_vec(word):
    return analogy_embeddings[word]

def analogy(a, b, c, vocab_words, top_n=3):
    """a is to b as c is to ?  (e.g., king:man :: queen:woman)"""
    result = get_vec(b) - get_vec(a) + get_vec(c)
    all_vecs = np.array([analogy_embeddings[w] for w in vocab_words])
    sims = cosine_similarity([result], all_vecs)[0]
    ranked = np.argsort(sims)[::-1]
    return [(vocab_words[i], sims[i]) for i in ranked[:top_n]]

tests = [
    ("man", "woman", "king", "queen"),
    ("man", "woman", "prince", "princess"),
    ("france", "paris", "england", "london"),
]

for a, b, c, expected in tests:
    results = analogy(a, b, c, analogy_vocab)
    print(f"{a} : {b} :: {c} : ?")
    for w, sim in results:
        marker = " ←" if w == expected else ""
        print(f"  {w:12s} {sim:.3f}{marker}")
    print()

---

## 10. Document Vectors from Word Embeddings

Embeddings represent words; documents are sequences of words. Common strategies to create **document vectors**:

| Method | Formula | Notes |
|--------|---------|-------|
| **Average** | $\frac{1}{n}\sum_i \mathbf{e}_{w_i}$ | Simple, works well |
| **Sum** | $\sum_i \mathbf{e}_{w_i}$ | Favors longer documents |
| **Weighted average** | $\sum_i \alpha_i \mathbf{e}_{w_i}$ | TF-IDF weights |
| **First word / [CLS]** | $\mathbf{e}_{w_1}$ | Used in BERT |

Average pooling is the most common approach for static embeddings. The resulting document vector can feed a classifier.

In [ ]:
class EmbeddingModel:
    def __init__(self, embeddings, vocab):
        self.vocab = vocab
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.embeddings = embeddings
        self.dim = embeddings.shape[1]

    def get(self, word):
        idx = self.word2idx.get(word)
        return self.embeddings[idx] if idx is not None else np.zeros(self.dim)

    def document_vector(self, text):
        tokens = tokenize(text)
        vecs = [self.get(t) for t in tokens if t in self.word2idx]
        return np.mean(vecs, axis=0) if vecs else np.zeros(self.dim)

emb_model = EmbeddingModel(embeddings, vocab)

for doc in ["the cat sat on the mat", "a car drove on the road"]:
    vec = emb_model.document_vector(doc)
    print(f"'{doc}'")
    print(f"  → vector shape {vec.shape}, norm={np.linalg.norm(vec):.3f}")

---

## 11. Text Classification with Embeddings

Convert each document to an averaged embedding, then train a classifier. This often outperforms TF-IDF on small datasets when embeddings capture semantic similarity between related words.

In [ ]:
# Animal vs vehicle classification using learned embeddings
train_corpus = [
    ("the cat sat on the mat", "animal"),
    ("the dog chased the cat", "animal"),
    ("a puppy played with a kitten", "animal"),
    ("the kitten played on the mat", "animal"),
    ("the mouse ran from the cat", "animal"),
    ("the dog sat on the rug", "animal"),
    ("a car drove down the road", "vehicle"),
    ("the truck drove on the road", "vehicle"),
    ("a car parked on the street", "vehicle"),
    ("the truck carried heavy load", "vehicle"),
]

# Rebuild embeddings on expanded corpus
expanded_corpus = [t for t, _ in train_corpus] + corpus
cooc_exp, vocab_exp = build_cooccurrence(expanded_corpus, window_size=2)
emb_exp, _ = svd_embeddings(cooc_exp, dim=16)
model_exp = EmbeddingModel(emb_exp, vocab_exp)

texts = [t for t, l in train_corpus]
labels = [l for t, l in train_corpus]
X = np.array([model_exp.document_vector(t) for t in texts])

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.3, random_state=42, stratify=labels
)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Embedding + logistic regression accuracy: {acc:.2f}")

# Test on new sentences
test_sentences = [
    ("the cat chased the mouse", "animal"),
    ("a truck drove fast on road", "vehicle"),
]
print("\nPredictions on new sentences:")
for sent, true_label in test_sentences:
    vec = model_exp.document_vector(sent).reshape(1, -1)
    pred = clf.predict(vec)[0]
    print(f"  '{sent}' → {pred} (actual: {true_label})")

---

## 12. Pre-trained Embeddings

Training embeddings from scratch requires large corpora (billions of words). **Pre-trained embeddings** learned on Wikipedia, Google News, or Common Crawl are available for direct use:

| Embeddings | Dimensions | Training Data | Notes |
|-----------|-----------|---------------|-------|
| **Word2Vec Google News** | 300 | 100B words | Most cited; English only |
| **GloVe** | 50–300 | Wikipedia + Gigaword | Multiple sizes available |
| **FastText** | 300 | Wikipedia | Subword info; handles OOV |

Using pre-trained embeddings:
1. Download the embedding file (word → vector mapping).
2. Build an embedding matrix aligned with your vocabulary.
3. Words in your vocab get pre-trained vectors; OOV words get random initialization.
4. Optionally **fine-tune** embeddings during downstream training.

Pre-trained embeddings provide an immediate boost when your task-specific data is limited.

---

## 13. FastText and Subword Information

Static embeddings assign one vector per word — OOV words get nothing. **FastText** (Facebook, 2016) represents each word as the **sum of its character n-gram vectors**:

*"where"* → character n-grams: `<wh`, `whe`, `her`, `ere`, `re>`, plus the word itself.

The vector for *"where"* is the sum of its n-gram vectors. A misspelling like *"wherre"* shares most n-grams with *"where"* and gets a similar vector. This handles:
- Out-of-vocabulary words
- Morphologically rich languages (Finnish, Turkish)
- Typos and spelling variations

FastText conceptually bridges word-level embeddings and the subword tokenization used in modern Transformers.

In [ ]:
def char_ngrams(word, min_n=3, max_n=5):
    word = f"<{word}>"
    ngrams = set()
    for n in range(min_n, max_n + 1):
        for i in range(len(word) - n + 1):
            ngrams.add(word[i:i+n])
    return ngrams

for word in ["cat", "cats", "category"]:
    grams = char_ngrams(word)
    print(f"{word:10s} → {sorted(grams)[:6]}... ({len(grams)} n-grams)")

cat_grams = char_ngrams("cat")
cats_grams = char_ngrams("cats")
overlap = cat_grams & cats_grams
print(f"\nShared n-grams between 'cat' and 'cats': {sorted(overlap)}")

---

## 14. Static vs Contextual Embeddings

Word2Vec, GloVe, and FastText produce **static embeddings** — one fixed vector per word regardless of context. *"Bank"* in *"river bank"* and *"bank account"* get the same vector.

**Contextual embeddings** (ELMo, BERT, GPT) produce different vectors for the same word in different contexts. The representation depends on surrounding words through deep neural networks.

| Type | Example | Same word, different context |
|------|---------|------------------------------|
| **Static** | Word2Vec, GloVe | Same vector always |
| **Contextual** | BERT, ELMo | Different vector per usage |

Contextual embeddings solved the limitation that static embeddings cannot disambiguate polysemous words. They are the standard in modern NLP — but static embeddings remain useful for lightweight applications, interpretability, and teaching.

In [ ]:
# Static embedding: "bank" has one vector regardless of context
print("Static embeddings (Word2Vec-style):")
print('  "bank" in "river bank"     → same vector e_bank')
print('  "bank" in "savings bank"   → same vector e_bank')
print()
print("Contextual embeddings (BERT-style):")
print('  "bank" in "river bank"     → vector e_1 (geographic sense)')
print('  "bank" in "savings bank"   → vector e_2 (financial sense)')
print('  e_1 ≠ e_2 — context determines representation')

---

## 15. Visualizing Embeddings

High-dimensional embeddings are projected to 2D for visualization using **PCA** (linear) or **t-SNE** (nonlinear). Clusters reveal semantic groupings — animals, vehicles, countries — and validate that the embedding space captures meaningful structure.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(analogy_embeddings_array := np.array(
    [analogy_embeddings[w] for w in analogy_vocab]
))

fig, ax = plt.subplots(figsize=(9, 7))
colors = {"king": "red", "queen": "red", "prince": "salmon", "princess": "salmon",
          "man": "blue", "woman": "blue", "boy": "lightblue", "girl": "lightblue",
          "paris": "green", "france": "green", "london": "orange", "england": "orange"}

for word, vec in zip(analogy_vocab, emb_2d):
    ax.scatter(vec[0], vec[1], s=100, color=colors.get(word, "gray"))
    ax.annotate(word, vec, textcoords="offset points", xytext=(6, 4), fontsize=11)

ax.set_title("Embedding space (PCA projection)")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 16. Embeddings vs TF-IDF

| Aspect | TF-IDF | Word Embeddings |
|--------|--------|----------------|
| **Representation** | Sparse, high-dim | Dense, low-dim (50–300) |
| **Semantic similarity** | None | Captured in vector space |
| **OOV handling** | `<UNK>` or zero | FastText subwords; contextual models |
| **Training data needed** | Task corpus only | Large corpus (or pre-trained) |
| **Interpretability** | Word weights are readable | Dimensions are not human-readable |
| **Compute** | Fast | Heavier (especially contextual) |
| **Best for** | Baselines, search, small data | Semantic tasks, transfer learning |

In practice: start with TF-IDF as a baseline. Use pre-trained embeddings when semantic similarity matters or labeled data is scarce. Use contextual embeddings (BERT) when word sense disambiguation is critical.

---

## 17. Best Practices

**Use pre-trained embeddings when possible.** Training from scratch needs gigabytes of text; pre-trained vectors are free and high quality.

**Align vocabulary carefully.** Your tokenizer vocabulary must map correctly to the embedding matrix rows.

**Handle OOV words.** Use FastText subwords, map to `<UNK>`, or initialize randomly and fine-tune.

**Average pooling is a strong baseline.** Mean of word vectors + logistic regression competes with TF-IDF on many tasks.

**Fine-tune for your domain.** Medical or legal text may need domain-specific embeddings; general embeddings miss specialized terminology.

**Consider contextual models for ambiguity.** If the same word has different meanings in your task, static embeddings are a limitation.

---

## 18. Summary

**Word embeddings** map words to dense, low-dimensional vectors where semantic similarity corresponds to geometric proximity. They arise from the distributional hypothesis — words in similar contexts get similar vectors.

**Co-occurrence matrices + SVD**, **Word2Vec** (skip-gram/CBOW), and **GloVe** are the main learning methods. Embeddings support **cosine similarity search**, **vector arithmetic** (king − man + woman ≈ queen), and **document classification** via averaging.

**Pre-trained embeddings** (Word2Vec, GloVe, FastText) provide immediate value on downstream tasks. **FastText** adds subword information for OOV handling. **Contextual embeddings** (BERT, GPT) extend the idea by producing context-dependent representations — the foundation of modern NLP.

Embeddings bridge the gap between discrete words and continuous mathematics, enabling neural networks to process language with rich semantic structure.